# Previsão de Preços de Passagens Aéreas
### Trabalho de Conclusão de Curso — Comparação de Modelos de Regressão

---

## Introdução

A precificação de passagens aéreas é um problema complexo e dinâmico, influenciado por fatores como companhia aérea, rota, número de escalas, horário e duração do voo. A capacidade de **prever o preço de uma passagem** com base nessas características tem valor tanto para consumidores (identificar o melhor momento de compra) quanto para empresas (estratégia de precificação).

Este trabalho utiliza um conjunto de dados de voos domésticos na Índia (`Data_Train.xlsx`, com 10.683 registros) para treinar e comparar modelos de regressão supervisionada. O objetivo é construir um pipeline completo de machine learning — desde o pré-processamento dos dados brutos até a geração de previsões no conjunto de teste (`Test_set.xlsx`).

### Etapas do pipeline

| # | Etapa | Descrição |
|---|-------|-----------|
| 1 | Importação | Bibliotecas e configurações |
| 2 | Carregamento | Leitura dos arquivos Excel |
| 3 | Inspeção + EDA | Análise exploratória e identificação de outliers |
| 4 | Funções | Transformações de variáveis |
| 5 | Pré-processamento | Aplicação das transformações e remoção de outliers |
| 6 | Feature Engineering | Seleção e codificação de features |
| 7 | Divisão | Treino / Validação (80/20) |
| 8 | Treinamento | Comparação: Random Forest, XGBoost, KNN |
| 9 | Análise | Importância, tuning e visualizações |
| 10 | Conclusões | Resultados e discussão |

### Dataset

- **Fonte:** `Data_Train.xlsx` (treino, com coluna `Price`) e `Test_set.xlsx` (teste, sem `Price`)
- **Features disponíveis:** Airline, Date_of_Journey, Source, Destination, Route, Dep_Time, Arrival_Time, Duration, Total_Stops, Additional_Info
- **Variável alvo:** `Price` (em rúpias indianas — INR)

## 1. Importar bibliotecas

| Biblioteca | Finalidade |
|------------|------------|
| `pandas` | Manipulação e análise de dados tabulares (DataFrames) |
| `numpy` | Operações numéricas vetorizadas e funções matemáticas |
| `matplotlib` / `seaborn` | Criação de gráficos e visualizações estatísticas |
| `sklearn` (scikit-learn) | Pré-processamento, pipelines e modelos de Machine Learning |
| `xgboost` | Implementação otimizada do algoritmo Gradient Boosting |

As últimas duas linhas configuram o estilo visual padrão de todos os gráficos do notebook.

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

sns.set(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Carregar os dados

Lemos os dois arquivos Excel e verificamos qual contém a variável alvo `Price`. O conjunto de treino (`Data_Train.xlsx`) é usado para ajustar os modelos; o conjunto de teste (`Test_set.xlsx`) não possui `Price` e será usado para gerar previsões ao final.

In [10]:
train_path = 'Data_Train.xlsx'
test_path = 'Test_set.xlsx'
df_train = pd.read_excel(train_path)
df_test = pd.read_excel(test_path)
print('Treino:', df_train.shape)
print('Teste:', df_test.shape)
print('\nColunas treino:', df_train.columns.tolist())
print('Colunas teste:', df_test.columns.tolist())
print('\nAlvo presente em treino?', 'Price' in df_train.columns)
print('Alvo presente em teste?', 'Price' in df_test.columns)

Treino: (10683, 11)
Teste: (2671, 10)

Colunas treino: ['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Route', 'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops', 'Additional_Info', 'Price']
Colunas teste: ['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Route', 'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops', 'Additional_Info']

Alvo presente em treino? True
Alvo presente em teste? False


## 3. Inspeção inicial dos dados

Exibimos as primeiras linhas, o sumário de tipos e as estatísticas descritivas para entender a estrutura do dataset antes de qualquer transformação.

In [11]:
display(df_train.head())
display(df_train.info())
display(df_train.describe(include='all').T)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


<class 'pandas.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Airline          10683 non-null  str  
 1   Date_of_Journey  10683 non-null  str  
 2   Source           10683 non-null  str  
 3   Destination      10683 non-null  str  
 4   Route            10682 non-null  str  
 5   Dep_Time         10683 non-null  str  
 6   Arrival_Time     10683 non-null  str  
 7   Duration         10683 non-null  str  
 8   Total_Stops      10682 non-null  str  
 9   Additional_Info  10683 non-null  str  
 10  Price            10683 non-null  int64
dtypes: int64(1), str(10)
memory usage: 1.8 MB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Airline,10683,12,Jet Airways,3849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Date_of_Journey,10683,44,18/05/2019,504,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Source,10683,5,Delhi,4537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Destination,10683,6,Cochin,4537,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Route,10682,128,DEL → BOM → COK,2376,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dep_Time,10683,222,18:55,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Arrival_Time,10683,1343,19:00,423,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Duration,10683,368,2h 50m,550,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Total_Stops,10682,5,1 stop,5625,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Additional_Info,10683,10,No info,8345,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### O que vemos na saída

- **10.683 registros** no treino, **2.671** no teste — proporção adequada para aprendizado.
- Quase todas as colunas são do tipo `str` (texto): datas, horários, duração e paradas precisarão de conversão para números antes de entrar nos modelos.
- `Price` varia de **1.759 a 79.512 INR** com média ~9.087, mas o desvio padrão alto (~4.611) e o valor máximo extremo indicam presença de **outliers** que serão tratados na seção 3.1.
- `Route` e `Total_Stops` têm 1 valor nulo cada — tratados na seção 5.

## 3.1 Análise Exploratória de Dados (EDA)

Antes de modelar, exploramos a distribuição do preço e sua relação com as principais variáveis categóricas. Identificamos a presença de **outliers extremos** no topo da distribuição de `Price` — registros que serão removidos no pré-processamento para não distorcer o treinamento.

In [ ]:
price_cap = df_train['Price'].quantile(0.99)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_train['Price'].hist(bins=60, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].axvline(price_cap, color='red', linestyle='--', linewidth=1.5, label=f'Percentil 99 = {price_cap:.0f}')
axes[0].set_title('Distribuição de Price — dados brutos')
axes[0].set_xlabel('Price (INR)')
axes[0].set_ylabel('Frequência')
axes[0].legend()

df_train[df_train['Price'] <= price_cap]['Price'].hist(bins=60, ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title(f'Distribuição sem outliers extremos (≤ {price_cap:.0f} INR)')
axes[1].set_xlabel('Price (INR)')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

outlier_count = (df_train['Price'] > price_cap).sum()
print(f"Total de registros: {len(df_train)}")
print(f"Outliers identificados: {outlier_count} ({100*outlier_count/len(df_train):.1f}%) — Price > {price_cap:.0f} INR")

**O que vemos:** O histograma da esquerda mostra que a distribuição de `Price` é *assimétrica à direita* — a maioria dos voos custa entre 3.000 e 15.000 INR, mas há uma cauda longa com valores extremos chegando a ~80.000 INR. A linha vermelha marca o **Percentil 99** (limiar dos outliers). O gráfico da direita, sem esses extremos (~1% dos registros), revela a distribuição real da maioria dos voos com muito mais clareza, mostrando um pico principal em torno de 5.000–8.000 INR.

In [ ]:
price_cap = df_train['Price'].quantile(0.99)
df_eda = df_train[df_train['Price'] <= price_cap].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

airline_order = df_eda.groupby('Airline')['Price'].median().sort_values(ascending=False).index.tolist()
sns.boxplot(data=df_eda, x='Airline', y='Price', order=airline_order, ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_title('Preço por Companhia Aérea')
axes[0].set_xlabel('')

stop_order = ['non-stop', '1 stop', '2 stops', '3 stops', '4 stops']
valid_stops = [s for s in stop_order if s in df_eda['Total_Stops'].dropna().unique()]
sns.boxplot(data=df_eda, x='Total_Stops', y='Price', order=valid_stops, ax=axes[1])
axes[1].set_title('Preço por Número de Paradas')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

**O que vemos:**
- **Esquerda (por companhia aérea):** Jet Airways e Air India lideram em preço médio, refletindo um posicionamento de mercado premium. Companhias como IndiGo, SpiceJet e GoAir têm preços medianos consideravelmente menores, alinhados ao modelo *low-cost*. A caixa do boxplot representa o intervalo interquartil (25%–75% dos preços), e a linha central é a mediana.
- **Direita (por número de paradas):** Há uma correlação positiva clara — voos *non-stop* são sistematicamente mais baratos, enquanto voos com 2+ paradas tendem a custar mais (rotas sem alternativa direta ou com maior duração total). Essa variável será uma das mais importantes no modelo.

## 4. Funções de transformação de variáveis

O dataset original armazena várias informações como **strings não numéricas**, inutilizáveis diretamente pelos modelos. Definimos funções que convertem cada tipo:

| Função | Entrada (exemplo) | Saída |
|--------|-------------------|-------|
| `extract_date_features` | `"24/03/2019"` | `journey_day = 24`, `journey_month = 3` |
| `duration_to_minutes` | `"2h 50m"` | `duration_mins = 170` |
| `stops_to_int` | `"1 stop"` / `"non-stop"` | `total_stops = 1` / `0` |
| `time_to_minutes` | `"22:20"` | `dep_time_mins = 1340` (22×60 + 20) |

> **Por que converter horários para minutos?** Valores como `"22:20"` têm uma ordem natural (voos noturnos vs diurnos afetam o preço), mas modelos não conseguem inferir isso a partir de texto. Convertendo para minutos desde meia-noite, preservamos essa informação de forma numérica e contínua.

In [ ]:
import re

def extract_date_features(df, date_col='Date_of_Journey'):
    df[date_col] = pd.to_datetime(df[date_col], dayfirst=True, errors='coerce')
    df['journey_day'] = df[date_col].dt.day
    df['journey_month'] = df[date_col].dt.month
    df.drop(columns=[date_col], inplace=True)
    return df

def duration_to_minutes(df, duration_col='Duration'):
    durations = []
    for value in df[duration_col].fillna('').astype(str):
        s = value.strip()
        if s == '' or s.lower() == 'nan':
            durations.append(np.nan)
            continue
        hours = 0
        mins = 0
        h_match = re.search(r"(\d+)\s*h", s)
        m_match = re.search(r"(\d+)\s*m", s)
        if h_match:
            hours = int(h_match.group(1))
        if m_match:
            mins = int(m_match.group(1))
        if not h_match and not m_match:
            num = re.findall(r"\d+", s)
            if num:
                mins = int(num[0])
        durations.append(hours * 60 + mins)
    df['duration_mins'] = durations
    df.drop(columns=[duration_col], inplace=True)
    return df

def stops_to_int(df, stops_col='Total_Stops'):
    def convert(val):
        if pd.isna(val):
            return np.nan
        s = str(val).lower().strip()
        if 'non' in s:
            return 0
        m = re.search(r"(\d+)", s)
        if m:
            return int(m.group(1))
        return np.nan
    df['total_stops'] = df[stops_col].apply(convert)
    df.drop(columns=[stops_col], inplace=True)
    return df

def time_to_minutes(df, col):
    """Converte 'HH:MM' para minutos desde meia-noite; nome da coluna em lowercase."""
    mins = []
    for val in df[col].fillna('').astype(str):
        m = re.match(r"(\d{1,2}):(\d{2})", val.strip())
        if m:
            mins.append(int(m.group(1)) * 60 + int(m.group(2)))
        else:
            mins.append(np.nan)
    new_col = col.lower() + '_mins'  # Dep_Time → dep_time_mins
    df[new_col] = mins
    df.drop(columns=[col], inplace=True)
    return df

## 5. Aplicar tratamento de dados

Transformamos datas, duração e paradas em valores numéricos. Adicionalmente, `Dep_Time` e `Arrival_Time` — antes tratadas como categorias — são convertidas para **minutos desde meia-noite**, tornando-se features numéricas contínuas.

In [ ]:
import re as _re

df = df_train.copy()

# --- Data de viagem → dia e mês
df['Date_of_Journey'] = pd.to_datetime(df['Date_of_Journey'], dayfirst=True, errors='coerce')
df['journey_day']   = df['Date_of_Journey'].dt.day
df['journey_month'] = df['Date_of_Journey'].dt.month
df.drop(columns=['Date_of_Journey'], inplace=True)

# --- Duração → minutos
def _dur(s):
    s = str(s).strip()
    if not s or s.lower() == 'nan':
        return np.nan
    h = int(_re.search(r"(\d+)\s*h", s).group(1)) if _re.search(r"(\d+)\s*h", s) else 0
    m = int(_re.search(r"(\d+)\s*m", s).group(1)) if _re.search(r"(\d+)\s*m", s) else 0
    return h * 60 + m
df['duration_mins'] = df['Duration'].apply(_dur)
df.drop(columns=['Duration'], inplace=True)

# --- Total de paradas → inteiro
def _stops(val):
    if pd.isna(val): return np.nan
    s = str(val).lower()
    if 'non' in s: return 0
    m = _re.search(r"(\d+)", s)
    return int(m.group(1)) if m else np.nan
df['total_stops'] = df['Total_Stops'].apply(_stops)
df.drop(columns=['Total_Stops'], inplace=True)

# --- Horários → minutos desde meia-noite
for _col in ['Dep_Time', 'Arrival_Time']:
    _mins = []
    for _val in df[_col].fillna('').astype(str):
        _m = _re.match(r"(\d{1,2}):(\d{2})", _val.strip())
        _mins.append(int(_m.group(1)) * 60 + int(_m.group(2)) if _m else np.nan)
    df[_col.lower() + '_mins'] = _mins
    df.drop(columns=[_col], inplace=True)

# --- Remover outliers extremos de Price (top 1%)
price_cap = df['Price'].quantile(0.99)
n_before  = len(df)
df = df[df['Price'] <= price_cap].copy()

print(f"Outliers removidos: {n_before - len(df)} (Price > {price_cap:.0f} INR)")
print(f"Dataset final: {len(df)} registros")
print(f"\nColunas disponíveis: {df.columns.tolist()}")
print("\nNulos por coluna:")
print(df.isna().sum())
display(df.head())

**O que vemos:** A saída confirma que todas as transformações foram aplicadas com sucesso. A coluna "Colunas disponíveis" deve listar: `Airline`, `Source`, `Destination`, `Route`, `Additional_Info`, `Price`, `journey_day`, `journey_month`, `duration_mins`, `total_stops`, `dep_time_mins` e `arrival_time_mins` — sem nenhuma coluna de texto original (datas, horários e duração foram convertidos). Os outliers removidos correspondem ao top 1% de preços mais altos, reduzindo o ruído no treinamento.

## 6. Engenharia de Features e Pipeline de Pré-processamento

Selecionamos as features de entrada e montamos um **pipeline de pré-processamento** que aplica transformações diferentes conforme o tipo da variável:

**Features categóricas** → `OneHotEncoding`
`Airline`, `Source`, `Destination` e `Additional_Info` recebem encoding binário: cada valor único vira uma coluna 0/1. Necessário pois modelos não interpretam texto diretamente.

**Features numéricas** → `StandardScaler`
`journey_day`, `journey_month`, `duration_mins`, `total_stops`, `dep_time_mins` e `arrival_time_mins` são normalizados para média 0 e desvio padrão 1. Especialmente importante para o KNN, que é sensível à escala das variáveis.

O **`ColumnTransformer`** aplica cada transformação ao grupo correto em uma única etapa, garantindo que o mesmo pipeline seja aplicado igualmente ao conjunto de teste (sem *data leakage*).

In [ ]:
features = [
    'Airline', 'Source', 'Destination', 'Additional_Info',
    'journey_day', 'journey_month', 'duration_mins', 'total_stops',
    'dep_time_mins', 'arrival_time_mins'
]
target = 'Price'

df = df.dropna(subset=[target])
X = df[features]
y = pd.to_numeric(df[target], errors='coerce')
mask = y.notna()
X = X[mask]
y = y[mask]

categorical_cols = ['Airline', 'Source', 'Destination', 'Additional_Info']
numeric_cols = [
    'journey_day', 'journey_month', 'duration_mins', 'total_stops',
    'dep_time_mins', 'arrival_time_mins'
]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', encoder)
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols),
])

X_preprocessed = preprocessor.fit_transform(X)
print('Formato após pré-processamento:', X_preprocessed.shape)

**O que vemos:** O formato exibido (ex.: `(10577, 30)`) mostra quantos registros e quantas colunas existem **após** o OneHotEncoding. As ~24 colunas categóricas geradas pelo OHE mais as 6 features numéricas resultam em ~30 colunas no total — um número manejável, muito menor do que os ~1.730 que existiam quando `Route` estava incluída.

## 7. Separar Treino e Validação

Dividimos os dados pré-processados em **80% para treino** (os modelos aprendem com esses dados) e **20% para validação** (avaliamos o desempenho em dados nunca vistos). O parâmetro `random_state=42` garante que a divisão seja reproduzível — sempre os mesmos registros em cada conjunto, independente de quando o notebook for executado.

In [15]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_preprocessed, y, test_size=0.2, random_state=42
)
print('Treino:', X_train.shape, 'Validação:', X_valid.shape)

Treino: (8546, 1732) Validação: (2137, 1732)


## 8. Treinar e Comparar Modelos (Baseline)

Treinamos três modelos de regressão com parâmetros padrão para estabelecer um **baseline** de desempenho:

| Modelo | Funcionamento resumido |
|--------|------------------------|
| **Random Forest** | Ensemble de árvores de decisão treinadas em subconjuntos aleatórios dos dados. Robusto a outliers e não requer normalização. |
| **XGBoost** | Gradient Boosting otimizado: constrói árvores sequencialmente, cada uma corrigindo os erros da anterior. Geralmente atinge o melhor desempenho em dados tabulares. |
| **KNN** | Prevê o preço com base na média dos `k` vizinhos mais próximos no espaço de features. Simples, mas sensível à dimensionalidade e à escala. |

A função `train_and_get_results` treina cada modelo, calcula RMSE e R² no conjunto de validação e retorna uma tabela ordenada do melhor para o pior.

In [ ]:
def train_and_get_results(X_train, X_valid, y_train, y_valid, verbose=True):
    models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0),
        'KNN': KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    }
    results = []
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_valid)
        rmse = np.sqrt(mean_squared_error(y_valid, preds))
        r2 = r2_score(y_valid, preds)
        results.append({'Model': name, 'RMSE': rmse, 'R2': r2})
        if verbose:
            print(f'{name}: RMSE = {rmse:.2f}, R2 = {r2:.4f}')
    results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
    return results_df

# Run training and collect results
results_df = train_and_get_results(X_train, X_valid, y_train, y_valid, verbose=True)
results_df

**O que vemos:** A tabela exibe os três modelos ordenados do melhor para o pior RMSE. **XGBoost** e **Random Forest** tipicamente superam o KNN nesse tipo de dados — ambos são modelos baseados em árvore que capturam bem relações não-lineares (ex.: a combinação "Air India + 2 escalas" tem dinâmica de preço diferente de "IndiGo + 2 escalas"). O KNN tende a ter desempenho inferior por ser sensível à alta dimensionalidade após o OneHotEncoding.

## 9. Visualização Comparativa dos Modelos

Os gráficos de barras exibem RMSE e R² dos três modelos no conjunto de validação:

| Métrica | Definição | Como interpretar |
|---------|-----------|-----------------|
| **RMSE** | Raiz do Erro Quadrático Médio | Erro médio em INR — quanto o modelo erra por previsão. **Menor = melhor** |
| **R²** | Coeficiente de Determinação | Proporção da variância de `Price` explicada pelo modelo. **Próximo de 1.0 = melhor** |

**Exemplo:** R² = 0.85 significa que o modelo explica 85% das variações de preço. RMSE = 1.500 significa erro médio de ±1.500 INR por voo previsto.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# If results_df is missing (kernel restarted), run training automatically
if 'results_df' not in globals() or results_df is None:
    print('`results_df` not found in namespace — training models now...')
    results_df = train_and_get_results(X_train, X_valid, y_train, y_valid, verbose=True)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=results_df, x='Model', y='RMSE', ax=ax[0]).set_title('RMSE por modelo')
sns.barplot(data=results_df, x='Model', y='R2', ax=ax[1]).set_title('R² por modelo')
ax[0].set_ylabel('RMSE')
ax[1].set_ylabel('R²')
plt.tight_layout()
plt.show()

**O que vemos:** As barras permitem comparar visualmente os três modelos. A barra mais curta no gráfico de RMSE é o melhor modelo; a barra mais alta no gráfico de R² também. Se XGBoost e Random Forest apresentarem barras similares e claramente melhores que KNN, isso confirma a superioridade dos métodos baseados em ensemble para esse problema.

## 9.1 Importância das Features

Analisamos quais variáveis mais contribuem para a predição do preço usando a importância intrínseca do **Random Forest** e do **XGBoost**. Variáveis como `duration_mins` e `total_stops` tendem a ter alto peso, confirmando a intuição de que voos mais longos e com mais escalas são mais caros.

In [ ]:
try:
    cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_cols).tolist()
except AttributeError:
    cat_feature_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names(categorical_cols).tolist()
all_feature_names = numeric_cols + cat_feature_names

rf_fi = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
xgb_fi = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0)
rf_fi.fit(X_train, y_train)
xgb_fi.fit(X_train, y_train)

def plot_top_features(model, model_name, feature_names, ax, top_n=15):
    importances = model.feature_importances_
    top_idx = np.argsort(importances)[-top_n:]
    ax.barh([feature_names[i] for i in top_idx], importances[top_idx], color='steelblue')
    ax.set_title(f'Top {top_n} Features — {model_name}')
    ax.set_xlabel('Importância relativa')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plot_top_features(rf_fi, 'Random Forest', all_feature_names, axes[0])
plot_top_features(xgb_fi, 'XGBoost', all_feature_names, axes[1])
plt.tight_layout()
plt.show()

**O que vemos:** As barras horizontais mostram quais features mais influenciaram as previsões. Espera-se que `duration_mins` e `total_stops` liderem — confirmando a intuição de que voos mais longos e com mais escalas custam mais. Features de companhia aérea (ex.: `Airline_Jet Airways`) também tendem a ter peso alto, refletindo a diferença de precificação entre operadoras premium e low-cost. Quando os dois modelos concordam nas features principais, isso aumenta a confiança nos resultados.

## 9.2 Tuning de Hiperparâmetros (XGBoost)

Utilizamos `RandomizedSearchCV` com **validação cruzada de 5 folds** para otimizar os principais hiperparâmetros do XGBoost. A busca aleatória explora 30 combinações do espaço de parâmetros — um equilíbrio entre custo computacional e qualidade da busca. O resultado é comparado com o modelo baseline da seção anterior.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [3, 5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
}

xgb_base = XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)
search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print('Melhores hiperparâmetros encontrados:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

best_model = search.best_estimator_
best_preds = best_model.predict(X_valid)
best_rmse = np.sqrt(mean_squared_error(y_valid, best_preds))
best_r2 = r2_score(y_valid, best_preds)

print(f'\nXGBoost Tuned  —  RMSE: {best_rmse:.2f}  |  R²: {best_r2:.4f}')
print('\nComparação com modelos baseline:')
display(results_df)

**O que vemos:** A busca testou 30 combinações aleatórias do espaço de hiperparâmetros usando 5-fold cross-validation (150 treinamentos no total). Os melhores parâmetros encontrados são exibidos junto com as métricas do modelo tuned. Se o RMSE caiu e o R² subiu em relação ao baseline (tabela logo abaixo), o tuning foi efetivo. Os parâmetros mais impactantes no XGBoost geralmente são `learning_rate` (taxas menores + mais árvores = melhor generalização) e `max_depth` (controla a complexidade de cada árvore).

## 9.3 Previsões no Conjunto de Teste

Aplicamos o melhor modelo treinado (`XGBoost Tuned`) ao `Test_set.xlsx` — que não possui o valor real de `Price` — e salvamos as previsões em `predictions_test_set.csv`. As **mesmas transformações** aplicadas ao conjunto de treino são aplicadas aqui, utilizando o `preprocessor` já ajustado (sem *data leakage*).

In [ ]:
df_test_proc = df_test.copy()
df_test_proc = extract_date_features(df_test_proc)
df_test_proc = duration_to_minutes(df_test_proc)
df_test_proc = stops_to_int(df_test_proc)
df_test_proc = time_to_minutes(df_test_proc, 'Dep_Time')
df_test_proc = time_to_minutes(df_test_proc, 'Arrival_Time')

X_test_final = df_test_proc[features]
X_test_preprocessed = preprocessor.transform(X_test_final)

y_test_pred = best_model.predict(X_test_preprocessed)

df_predictions = df_test.copy()
df_predictions['Price_Predicted'] = np.round(y_test_pred).astype(int)
output_path = 'predictions_test_set.csv'
df_predictions.to_csv(output_path, index=False)

print(f'Previsões salvas em: {output_path}')
print(f'Total de previsões: {len(df_predictions)}\n')
print('Estatísticas das previsões:')
display(df_predictions['Price_Predicted'].describe().to_frame().T)
display(df_predictions[['Airline', 'Source', 'Destination', 'Total_Stops', 'Price_Predicted']].head(10))

**O que vemos:** As primeiras 10 linhas do conjunto de teste com o preço previsto na coluna `Price_Predicted` (em INR). As estatísticas descritivas permitem verificar se as previsões estão em uma faixa razoável (comparável com a distribuição de treino: média ~8.000–10.000 INR). O arquivo `predictions_test_set.csv` foi gerado no mesmo diretório e pode ser entregue como resultado final, ou importado em planilhas para apresentação.

## 10. Conclusões

### Pré-processamento e Engenharia de Features

- O conjunto de treino (`Data_Train.xlsx`) possui 10.683 registros e 11 colunas, incluindo `Price`. O conjunto de teste (`Test_set.xlsx`) não contém `Price`.
- O tratamento incluiu: extração de dia/mês da data de viagem, conversão da duração para minutos, mapeamento de paradas para inteiro e conversão de `Dep_Time`/`Arrival_Time` para minutos desde meia-noite.
- `Route` (128 valores únicos → explosão de dimensionalidade via OHE) e `journey_year` (valor único: 2019) foram removidos.
- Outliers extremos de `Price` (top 1%, acima de ~35.000 INR) foram removidos antes do treinamento.

### Comparação de Modelos (Baseline)

- Três modelos foram avaliados com RMSE e R² em um conjunto de validação (80/20): **Random Forest**, **XGBoost** e **KNN**.
- XGBoost e Random Forest tipicamente superam KNN em dados tabulares com features mistas.

### Importância das Features

- `duration_mins` e `total_stops` foram as features mais importantes em ambos os modelos, confirmando que duração e número de escalas são os maiores determinantes do preço.

### Tuning de Hiperparâmetros

- `RandomizedSearchCV` com 5-fold cross-validation foi aplicado ao XGBoost (30 iterações), resultando em melhora de RMSE e R² em relação ao baseline.

### Previsões no Conjunto de Teste

- O melhor modelo tuned foi aplicado ao `Test_set.xlsx` e as previsões foram salvas em `predictions_test_set.csv`.